# Designing & Building AI Agents That Work and Scale
### 📘 INSTRUCTOR / MARKING-SCHEME COPY — fully completed
### A Hands-On Lab: From Prototype to Production Agent

**Goal of this lab:** by the end, you will have built a small but *real* AI agent,
piece by piece, following the same anatomy that separates a flashy demo from a
system you could actually trust in production:

> **Agent = Model + Instructions + Context + Tools + State + Guardrails + Evaluation + Observability**

We will build **one component at a time**, test it in isolation, and only
then bolt it onto the next one. This mirrors how real engineering teams
build agents: nobody writes the whole thing in one shot — you build a
reliable foundation and layer capability on top of it. By the last section
you will assemble everything into a single `MiniAgent` class, and finally
turn it into an interactive chatbot you can actually talk to.

**How to use this notebook**
- Run the cells top to bottom, in order — later sections depend on variables
  and functions defined in earlier ones, so skipping around will cause
  `NameError`s.
- Cells marked **✅ Completed** are for you to complete. There is a short hint
  above each one, and a ✅ **Reference solution** cell right after it — try
  first, peek only if you get stuck. Struggling with a TODO for a few
  minutes before looking at the answer is where most of the learning
  happens, so resist the urge to scroll down immediately.
- Cells marked **▶️ Run me** are fully provided and should just work — read
  them anyway, since they're doing most of the actual teaching.
- Every code cell is heavily commented. If a line of code doesn't make
  sense, look for a `#` comment directly above or beside it before asking
  for help — the answer is probably already there.

Let's get started.


---
## 0. Setup

Unlike some hosted providers, the model in this lab is served behind a plain
**OpenAI-compatible REST endpoint** — no vendor SDK, just `requests` and
JSON. This is a very common real-world setup, especially for self-hosted or
internal inference servers (this one is serving a Qwen model through a
vLLM-style API gateway), so it's a genuinely useful skill to know how to
talk to a model with nothing but raw HTTP.

**What "OpenAI-compatible" means:** many different companies and open-source
projects have adopted the same request/response *shape* that OpenAI's API
uses (a `messages` list, a `/v1/chat/completions` path, a JSON body with
fields like `temperature`). That means code written against one
OpenAI-compatible server usually works against another with just a URL and
API key change — which is exactly what's happening here.

Run the cell below to install what we need.


In [ ]:
# ▶️ Run me
# `requests` is Python's most widely used HTTP client library — it's how
# we'll send data to the model server and receive its reply back.
%pip install -q requests


In [ ]:
# ▶️ Run me — configuration
import requests   # for making HTTP calls to the model API
import json       # for encoding/decoding the JSON request and response bodies
import time       # we'll use this later for timing/observability

# The URL of the model server's chat-completions endpoint.
# This follows the OpenAI convention: POST a JSON body to a /v1/chat/completions path.
API_URL = "https://p36tc5pup6.execute-api.eu-west-1.amazonaws.com/staging//v1/chat/completions"

# Your API key. In a real project this would NEVER be hard-coded like this —
# it would live in an environment variable or secrets manager, and this
# notebook would read it with something like os.environ["API_KEY"]. We're
# hard-coding it here purely so everyone in the room has an identical,
# working setup with zero extra steps.
API_KEY = "sk-ZBrnpC9K9h7S8YCzQ36a6g"

# Which model to ask the server for. A single server can often host several
# models; this string tells it which one to route your request to.
MODEL = "Qwen3.6-35B-A3B"

# HTTP headers sent with every request:
#  - Content-Type tells the server we're sending JSON.
#  - Authorization proves who we are, using the standard "Bearer <token>" scheme.
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

print("Configured model:", MODEL)
print("Endpoint:", API_URL)


### The low-level building block: `chat_completion()`

Every single thing we build in this lab — instructions, context, tools,
memory, guardrails — ultimately boils down to sending a list of `messages`
to this one endpoint and reading back a reply. Let's write that one function
once, carefully, and reuse it everywhere, instead of copy-pasting the same
`requests.post(...)` call 20 times. This is the **DRY principle**
("Don't Repeat Yourself") in action — if the API ever changes, we fix it
in exactly one place.

**A quick primer on the `messages` format**, since you'll see it constantly
from here on: a conversation is a Python list of dictionaries, each with a
`role` (`"system"`, `"user"`, `"assistant"`, or `"tool"`) and `content`
(the text). For example:

```python
[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
]
```

The model reads this whole list every time and predicts what the next
`assistant` message should be.


In [ ]:
# ▶️ Run me
def chat_completion(messages, tools=None, temperature=1, max_tokens=8000, **kwargs):
    """Call the chat completions endpoint and return the parsed JSON response.

    This is the ONE function that actually talks to the model. Everything
    else in this notebook is built on top of it.

    Args:
        messages: list of {"role": ..., "content": ...} dicts (OpenAI format).
                  This is the entire conversation the model will see.
        tools:    optional list of tool schemas (OpenAI function-calling
                  format). Only needed once we get to Section 4 — leave as
                  None for plain text conversations.
        temperature: 0 = mostly deterministic/focused, 1 = more varied and
                  creative. Covered properly in Section 1.
        max_tokens: the hard cap on how long the reply is allowed to be
                  (measured in tokens, roughly ~0.75 words each).
        **kwargs: anything else you want to override in the request body
                  (e.g. top_p) without having to change this function's
                  signature every time.

    Returns:
        The full JSON response from the server, as a Python dict.
    """
    # Build the request body. These are the exact fields the server expects —
    # most of them map directly onto the sample cURL/requests call you were
    # given, just wrapped in a reusable function instead of copy-pasted.
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "top_p": 0.95,               # nucleus sampling: only consider the top 95% probability mass of next-word candidates
        "max_tokens": max_tokens,
        "presence_penalty": 0,       # >0 discourages repeating ANY previously-used token, regardless of how often
        "frequency_penalty": 0,      # >0 discourages repeating tokens IN PROPORTION to how often they've already appeared
        "stream": False,             # we want the whole reply back in one go, not streamed token-by-token
        "n": 1,                      # ask for exactly one completion (some APIs let you request several alternatives)
        "chat_template_kwargs": {"enable_thinking": False},  # server-specific: disables the model's internal "thinking" trace
        "reasoning_effort": "low",   # server-specific: how much extra reasoning compute to spend per request
    }
    if tools:
        payload["tools"] = tools

    # Let the caller override or add any field above without us having to
    # anticipate every possible option in this function's signature.
    payload.update(kwargs)

    # Send the POST request. json.dumps() turns our Python dict into a JSON string.
    response = requests.post(API_URL, headers=HEADERS, data=json.dumps(payload))

    # If the server returned an error status code (4xx/5xx), raise an
    # exception immediately instead of silently returning garbage — this
    # makes bugs much easier to spot while you're learning.
    response.raise_for_status()

    # Parse the JSON response body into a Python dict and return it.
    return response.json()


def get_message(response_json):
    """Pull out the assistant's message dict from a raw API response.

    The raw response has a fairly deep structure —
    response["choices"][0]["message"] — so this tiny helper saves us from
    retyping that path everywhere and makes the calling code more readable.
    """
    return response_json["choices"][0]["message"]


def get_text(response_json):
    """Pull out just the reply text (a string) from a raw API response.

    Use this whenever you just want the words the model said, with no
    tool-call metadata attached.
    """
    return get_message(response_json)["content"]


---
## 1. Model — "Language, reasoning & decision-making. But the model is not the product."

The **model** is the raw language-reasoning engine — the part everyone
thinks of first when they hear "AI agent." But on its own it's just a very
good autocomplete: it has no memory, no access to your systems, and no
concept of what it's *for*. Before we build anything on top of it, we need
to prove the engine turns on. Let's send the simplest possible request.


In [ ]:
# ▶️ Run me
# messages is a list with a single "user" turn — the minimum viable conversation.
response = chat_completion([
    {"role": "system", "content": "You are an expert MLOps assistant."},
    {"role": "user", "content": "In one short sentence, what is an AI agent?"},
])

# get_text() digs the actual reply string out of the (fairly nested) response JSON.
print(get_text(response))


If you saw a sentence printed above, your connection to the model works.
That's it — that's the whole "Model" component tested. Everything else in
this lab is about the *scaffolding* around this one HTTP call: giving the
model a job (Instructions), facts (Context), abilities (Tools), memory
(State), boundaries (Guardrails), a way to check it's working (Evaluation),
and a way to see what it did (Observability).

**About `temperature`:** this single number controls how "safe" vs.
"adventurous" the model's word choices are.
- `temperature=0` → close to deterministic. Good for factual Q&A, code, and
  anything where you want the same input to reliably give (roughly) the
  same output.
- `temperature=1` (or higher, if the server allows it) → much more varied
  and creative, at the cost of consistency. Good for brainstorming or
  creative writing, bad for anything where correctness matters.

### ✅ Completed 1.1
Write a small helper function `ask(prompt, temperature=1)` that:
1. Sends `prompt` as a single user message to `chat_completion`.
2. Returns just the reply text (a string, not the whole response dict).

Try it out by asking it something, then try again with `temperature=0` and
`temperature=1` on a creative prompt (e.g. "invent a name for a coffee shop")
and notice how the variety of answers changes if you run the same cell
twice at each setting.


In [ ]:
# ✅ SOLVED — 1.1
def ask(prompt, temperature=1):
    """Send a single-turn question to the model and return the reply text."""
    response = chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return get_text(response)

# At temperature=0, run this twice — the two answers should be identical or near-identical.
print(ask("invent a name for a coffee shop", temperature=0))
# At temperature=1, run this twice — expect different answers each time.
print(ask("invent a name for a coffee shop", temperature=1))


---
## 2. Instructions — "Purpose, boundaries, trusted sources, and when to involve a person."

Instructions are delivered as a **system message** — a special message with
`role: "system"` that sits before anything the user says, and shapes how
the model behaves for the *entire* conversation. This is where you define:
- **Purpose** — what job is this agent actually doing?
- **Boundaries** — what should it refuse to do or talk about?
- **Trusted sources** — where should it get facts from (we'll wire this up
  properly in Section 3)?
- **Escalation** — when should it stop and hand off to a human?

Without a system message, the model is a generic chatbot with no job. Think
of it as the difference between hiring someone and giving them a job
description, versus just putting them in a room and hoping for the best.

Let's build a system prompt for a concrete persona: a **university helpdesk
assistant** for a Computer Science department.


In [ ]:
# ▶️ Run me
# Triple-quoted strings let us write a multi-line system prompt cleanly.
# Notice the structure: Purpose, then Boundaries, then an escalation rule —
# this is a reusable pattern you can apply to almost any agent persona.
SYSTEM_PROMPT = """You are UniHelp, the CS department's helpdesk assistant.

Purpose: help students with course registration questions, lab access,
and where to find department resources.

Boundaries:
- You only answer questions about the CS department. For anything else
  (grades in other departments, personal/medical/financial issues), politely
  say it's outside your scope and suggest the right office.
- You never make up policies, deadlines, or fees. If you don't know, say so
  and suggest emailing cs-helpdesk@example.edu.
- If a student sounds distressed or mentions an emergency, tell them to
  contact campus security or student wellness immediately.
"""

def ask_with_instructions(prompt):
    """Ask a question with the SYSTEM_PROMPT persona applied."""
    response = chat_completion([
        {"role": "system", "content": SYSTEM_PROMPT},  # sets the persona for this whole exchange
        {"role": "user", "content": prompt},            # the actual question
    ])
    return get_text(response)

print(ask_with_instructions("How do I get access to the AI lab?"))


In [ ]:
# ▶️ Run me — see the boundary in action
# This question is deliberately OUT of scope for UniHelp (it's an economics
# grade, not a CS department matter). A well-written system prompt should
# make the model decline gracefully instead of guessing or making something up.
print(ask_with_instructions("Can you tell me my grade in ECON 101?"))


### ✅ Completed 2.1
Write your own system prompt for a **different persona** of your choice
(examples: a telecom customer-care bot, a library assistant, a hackathon
mentor bot). Include at minimum:
- A **purpose** sentence.
- At least one **boundary** (something it should refuse or redirect).
- At least one **escalation rule** (when it should hand off to a human).

Test it with two prompts: one it should answer normally, and one that should
trigger your boundary or escalation rule. If the model doesn't respect your
boundary, try being more explicit and concrete in the prompt — vague
instructions get vague compliance.


In [ ]:
# ✅ SOLVED — 2.1 (one possible answer)
MY_SYSTEM_PROMPT = """You are LibBot, the university library's assistant.

Purpose: help students find books, check opening hours, and renew loans.

Boundaries:
- You do not have access to real account data in this demo — never invent
  a due date or fine amount; say you can't confirm real account details.
- If asked about anything outside library services, redirect politely.

Escalation:
- If someone reports a lost library card or suspected fraud, tell them to
  visit the front desk in person rather than handling it in chat.
"""

def ask_custom(prompt):
    """Ask a question using MY_SYSTEM_PROMPT (the LibBot persona) instead of UniHelp."""
    response = chat_completion([
        {"role": "system", "content": MY_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ])
    return get_text(response)

# This one should be answered normally — it's squarely in scope.
print(ask_custom("What time does the library open on Saturdays?"))
# This one should trigger the escalation rule — watch for "front desk" or "in person" in the reply.
print(ask_custom("I think someone stole my library card, what do I do?"))


---
## 3. Context — "The minimum trustworthy information needed for this decision."

Models don't know your organization's specific facts — they were trained on
a broad snapshot of the internet, not your department's lab-access policy.
**Context** means giving the model *just enough* real, trustworthy
information to answer correctly — not everything you have, just what's
relevant to this particular question.

Why "minimum"? Two reasons:
1. **Cost & speed** — every extra sentence of context costs tokens (and
   therefore money and latency) on *every single request*.
2. **Accuracy** — stuffing irrelevant information into the prompt makes it
   easier for the model to get confused or cite the wrong fact. Precision
   beats volume.

We'll simulate this with a tiny knowledge base and a simple retrieval
function. In a real system, this retrieval step is usually a search over a
vector database or search index — this is a miniature version of what's
usually called **RAG (Retrieval-Augmented Generation)**: retrieve the
relevant facts first, then generate an answer grounded in them.


In [ ]:
# ▶️ Run me
# A tiny "knowledge base" — in production this might be thousands of
# documents in a vector database; here it's just a Python list so you can
# see exactly what's happening under the hood.
KNOWLEDGE_BASE = [
    {"id": "kb1", "topic": "lab access", "text": "The AI Lab (Room 204) is open 8am-8pm weekdays. Access requires a signed lab-use form from your course instructor, submitted to the CS office."},
    {"id": "kb2", "topic": "registration", "text": "Course registration for the next semester opens two weeks before the semester ends and closes one week after it starts, via the student portal."},
    {"id": "kb3", "topic": "office hours", "text": "The CS helpdesk office is open Mon-Fri, 9am-4pm, in the department building, ground floor."},
]

def retrieve_context(query, kb=KNOWLEDGE_BASE, top_k=1):
    """Very simple keyword-overlap retrieval.

    This is NOT real semantic search (a production system would use
    embeddings and vector similarity so it can match meaning, not just
    exact words) — but it's simple enough to see exactly how retrieval
    works, which is the whole point of this exercise.

    Args:
        query: the user's question, as plain text.
        kb: the list of knowledge base entries to search.
        top_k: how many of the best-matching entries to return.

    Returns:
        A list of the top_k knowledge base entries with the most word
        overlap with the query (entries with zero overlap are excluded).
    """
    query_words = set(query.lower().split())
    scored = []
    for entry in kb:
        # Count how many words the query and this KB entry have in common.
        overlap = len(query_words & set(entry["text"].lower().split()))
        scored.append((overlap, entry))
    # Sort by overlap score, highest first.
    scored.sort(key=lambda x: x[0], reverse=True)
    # Keep only the top_k entries, and only if they actually matched something (score > 0).
    return [entry for score, entry in scored[:top_k] if score > 0]

# Try it: this should surface the "lab access" entry (kb1).
retrieve_context("how do I get access to the lab")


### ✅ Completed 3.1
Write `ask_with_context(query)` that:
1. Calls `retrieve_context(query)` to get relevant knowledge base entries.
2. Builds a system message that includes those entries as trusted context
   (tell the model to only use this context and say "I don't know" if the
   answer isn't in it — this instruction is critical, without it the model
   will happily guess).
3. Sends the user's query and returns the model's answer.

Test it with a question that IS in the knowledge base, and one that ISN'T
(e.g. "what's the wifi password?") — the model should admit it doesn't know
for the second one instead of guessing (this is called avoiding
**hallucination** — the model confidently making something up).


In [ ]:
# ✅ SOLVED — 3.1
def ask_with_context(query):
    """Answer a query, grounded ONLY in facts retrieved from KNOWLEDGE_BASE."""
    context_entries = retrieve_context(query)

    # Turn the list of matched entries into a bullet-point string for the prompt.
    # If nothing matched, fall back to a placeholder so the prompt still reads sensibly.
    context_text = "\n".join(f"- {e['text']}" for e in context_entries) or "(no relevant context found)"

    # The key instruction here is "Answer ONLY using the context below" —
    # this is what stops the model from filling gaps with invented facts.
    system_msg = (
        "You are a CS department assistant. Answer ONLY using the context "
        "below. If the context does not contain the answer, say you don't "
        "know and suggest contacting the CS helpdesk.\n\n"
        f"Context:\n{context_text}"
    )

    response = chat_completion([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": query},
    ])
    return get_text(response)

# In the knowledge base -> should get a grounded, correct answer.
print(ask_with_context("how do I get access to the lab"))
print("---")
# NOT in the knowledge base -> should admit it doesn't know, not invent an answer.
print(ask_with_context("what's the wifi password?"))


---
## 4. Tools — "Search, query, validate, calculate, create, notify, act."

So far the model only *talks*. Tools let it *do* things: call a function,
hit an API, run a calculation, look something up in a database. The pattern
has three steps every time:

1. We tell the model what tools exist (name, description, and what
   arguments they take) using a JSON **schema**.
2. The model, given a user's message, decides *whether* a tool is needed
   and *which one*, and generates the arguments to call it with — but it
   does **not** run any code itself.
3. **Our code** actually executes the function, and sends the result back
   to the model so it can produce a final, natural-language answer.

This separation matters: the model proposes, your code disposes. That's
also exactly where Guardrails (Section 6) will plug in later — as a
checkpoint between step 2 and step 3.

Let's give our agent two tools: a calculator and a (mocked) course-seat
checker.

**Note:** this server exposes an OpenAI-compatible `tools` parameter, so the
mechanics below are identical to talking to any other OpenAI-style API —
only the transport (`requests` instead of an SDK) is different.


In [ ]:
# ▶️ Run me — the actual Python functions behind the tools
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression string, e.g. "2 + 2 * 3"."""
    try:
        # SAFETY NOTE: eval() executes arbitrary Python and is normally
        # dangerous to run on untrusted input. It's only acceptable here
        # because we FIRST whitelist which characters are allowed (digits,
        # operators, parentheses, spaces) — so nothing like "__import__('os')"
        # could ever get through. In a real system, use a proper math
        # expression parser (e.g. the `ast` module or a library like `numexpr`)
        # instead of eval(), even with this whitelist.
        allowed_chars = set("0123456789+-*/(). ")
        if not set(expression) <= allowed_chars:
            return "Error: expression contains disallowed characters."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# A mock "database" of course seat counts — in a real system this would be
# a live query against a student information system.
MOCK_SEATS = {"CS401": 3, "CS220": 0, "CS150": 12}

def check_seats(course_code: str) -> str:
    """Look up how many seats remain in a given course code."""
    seats = MOCK_SEATS.get(course_code.upper())
    if seats is None:
        return f"No such course code: {course_code}"
    return f"{course_code.upper()} has {seats} seat(s) remaining."

# This is the "menu" of tools we hand to the model. Each entry describes:
#   - name: how the model will refer to this tool
#   - description: helps the model decide WHEN to use it
#   - parameters: a JSON Schema describing what arguments it needs, and their types
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a basic arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "e.g. '2 + 2 * 3'"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_seats",
            "description": "Check how many seats remain in a given course code.",
            "parameters": {
                "type": "object",
                "properties": {"course_code": {"type": "string", "description": "e.g. 'CS401'"}},
                "required": ["course_code"],
            },
        },
    },
]

# A lookup from tool name -> the actual Python function to run. This is what
# lets our code translate "the model wants to call check_seats" into
# actually calling check_seats(...).
AVAILABLE_FUNCTIONS = {"calculate": calculate, "check_seats": check_seats}


In [ ]:
# ▶️ Run me — the tool-calling loop
def ask_with_tools(prompt, system_prompt=SYSTEM_PROMPT):
    """Ask a question, letting the model call tools if it decides to.

    This function makes UP TO TWO calls to the model:
      1st call: "here's the question and the tools you have — what do you want to do?"
      2nd call: only if a tool was used — "here's what the tool returned, now give a final answer."
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    # First call: pass `tools=TOOLS` so the model KNOWS these functions exist.
    response = chat_completion(messages, tools=TOOLS)
    msg = get_message(response)

    # Did the model ask to call one or more tools? If msg["tool_calls"] is
    # missing or empty, the model answered directly with no tool needed.
    tool_calls = msg.get("tool_calls")
    if tool_calls:
        # Add the model's own tool-call request to the conversation history —
        # the model needs to "see" its own request in the next call for context.
        messages.append(msg)

        for tool_call in tool_calls:
            fn_name = tool_call["function"]["name"]
            # Tool arguments come back as a JSON STRING, not a dict — we must parse them.
            fn_args = json.loads(tool_call["function"]["arguments"])
            print(f"  [tool call] {fn_name}({fn_args})")

            # THIS is the step where OUR code (not the model) actually runs the function.
            result = AVAILABLE_FUNCTIONS[fn_name](**fn_args)

            # Feed the tool's result back into the conversation with role="tool",
            # tagged with the same tool_call_id so the model can match it to its request.
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": result,
            })

        # Second call: now that the model has the tool result, ask it to
        # produce a final, natural-language answer.
        follow_up = chat_completion(messages, tools=TOOLS)
        return get_text(follow_up)

    # No tool was needed — the first response already contains the final answer.
    return msg["content"]

print(ask_with_tools("How many seats are left in CS401?"))


### ✅ Completed 4.1
Add a **third tool** called `word_count` that takes a string and returns how
many words it contains. You'll need to:
1. Write the Python function.
2. Add its schema to a new `TOOLS_V2` list (copy `TOOLS` and append yours).
3. Add it to a new `AVAILABLE_FUNCTIONS_V2` dict.
4. Test by asking something like *"How many words are in the sentence: the
   quick brown fox jumps over the lazy dog?"*

Tip: reuse `ask_with_tools`'s pattern, or copy it into a new function that
uses your v2 tools/functions.


In [ ]:
# ✅ SOLVED — 4.1
def word_count(text: str) -> str:
    """Count the words in `text` by splitting on whitespace."""
    return str(len(text.split()))

# {**dict, key: value} creates a NEW dict/list that's a copy of the original
# plus our addition, rather than mutating TOOLS / AVAILABLE_FUNCTIONS directly —
# this keeps the original v1 versions intact for comparison.
TOOLS_V2 = TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "word_count",
            "description": "Count how many words are in a piece of text.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
    }
]
AVAILABLE_FUNCTIONS_V2 = {**AVAILABLE_FUNCTIONS, "word_count": word_count}

def ask_with_tools_v2(prompt, system_prompt=SYSTEM_PROMPT):
    """Identical logic to ask_with_tools, just wired up to the v2 tool set."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    response = chat_completion(messages, tools=TOOLS_V2)
    msg = get_message(response)
    tool_calls = msg.get("tool_calls")
    if tool_calls:
        messages.append(msg)
        for tool_call in tool_calls:
            fn_name = tool_call["function"]["name"]
            fn_args = json.loads(tool_call["function"]["arguments"])
            result = AVAILABLE_FUNCTIONS_V2[fn_name](**fn_args)
            messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": result})
        follow_up = chat_completion(messages, tools=TOOLS_V2)
        return get_text(follow_up)
    return msg["content"]

print(ask_with_tools_v2("How many words are in the sentence: the quick brown fox jumps over the lazy dog?"))


---
## 5. State & Memory — "Where it is in the task; what it's authorised to retain."

So far every call has been **stateless** — the model has no memory of
previous turns. Ask it your name, then ask "what's my name?" in a fresh
call, and it will have no idea. Real conversations need **state**: a
running history the agent consults on every turn, plus rules about what to
keep vs. discard (you can't keep every message forever — context windows,
and your API bill, both have limits).


In [ ]:
# ▶️ Run me
class Conversation:
    """Wraps chat_completion() with a persistent message history, so the
    model can remember earlier turns in the same conversation."""

    def __init__(self, system_prompt, max_turns=6):
        self.system_prompt = system_prompt
        self.max_turns = max_turns  # how many user+assistant turn PAIRS to retain
        self.history = []  # list of {"role":..., "content":...} dicts, oldest first

    def _trim(self):
        """Keep only the most recent max_turns pairs, so history doesn't grow forever."""
        limit = self.max_turns * 2  # each "turn" is a user message + an assistant reply
        if len(self.history) > limit:
            # Python list slicing: [-limit:] keeps only the LAST `limit` items.
            self.history = self.history[-limit:]

    def ask(self, user_message):
        """Send a new message, using and updating the running conversation history."""
        # 1. Record the user's new message in history BEFORE calling the model,
        #    so it's included in what we send.
        self.history.append({"role": "user", "content": user_message})

        # 2. Build the full message list: system prompt + entire history so far.
        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        # 3. Call the model with the full context.
        response = chat_completion(messages)
        reply = get_text(response)

        # 4. Record the assistant's reply too, so future turns can see it.
        self.history.append({"role": "assistant", "content": reply})

        # 5. Trim old history if we've grown past max_turns.
        self._trim()
        return reply

convo = Conversation(SYSTEM_PROMPT)
print(convo.ask("My name is Amina and I'm a 2nd year student."))
print(convo.ask("What's my name and year?"))


Notice the second answer correctly recalls Amina's name — that's state
working. Without appending to `self.history`, each call would be a total
stranger to the last one. This is also why `max_turns` matters: every extra
turn you keep means every future request gets bigger (and slower, and more
expensive) — trimming is a real design decision, not just cleanup.

### ✅ Completed 5.1
The `_trim()` method above keeps the most recent turns, but it **always
keeps the oldest history entries too** if we never call `_trim()` correctly
inside a loop — trace through the code above and confirm for yourself when
`_trim()` actually gets called. Then, create a subclass `ConversationV2`
that also exposes a `summary()` method that asks the model to produce a
one-sentence summary of the conversation so far (useful for logging, or for
handing a conversation off to a human agent). Test it after a couple of
exchanges.


In [ ]:
# ✅ SOLVED — 5.1
class ConversationV2(Conversation):
    def summary(self):
        """Ask the model to summarize the conversation so far in one sentence."""
        # Flatten the history list into a readable "role: content" transcript.
        transcript = "\n".join(f"{m['role']}: {m['content']}" for m in self.history)

        # This is a completely separate, one-off call — it does NOT use
        # self.system_prompt, because summarizing is a different job from
        # being the persona itself.
        response = chat_completion([
            {"role": "system", "content": "Summarize the following conversation in one sentence."},
            {"role": "user", "content": transcript},
        ])
        return get_text(response)

convo2 = ConversationV2(SYSTEM_PROMPT)
convo2.ask("My name is Amina and I'm a 2nd year student.")
convo2.ask("I want to know how to get lab access.")
print(convo2.summary())


---
## 6. Guardrails — "Auth, permissions, approvals, escalation, auditability."

Guardrails are checks that sit **outside** the model's judgement — plain
Python code you control, that decides what the agent is actually allowed to
do. A model can be persuaded (via clever prompting), can misunderstand
instructions, or can simply make mistakes; guardrails are your safety net
that doesn't rely on the model "deciding" to behave.

We'll add two kinds of guardrail to our tool-calling agent:
1. **Input guardrail** — reject or flag disallowed input *before* it even
   reaches the model (cheaper, faster, and doesn't expose the model to
   attempted manipulation at all).
2. **Action guardrail** — some tool calls are sensitive enough that they
   require human approval before they're allowed to actually execute, even
   if the model wants to call them.


In [ ]:
# ▶️ Run me
# A (deliberately simple) keyword blocklist. Real systems typically layer
# several techniques here: keyword/regex filters, a separate classifier
# model, rate limiting, etc. — this is the simplest possible version so you
# can see the pattern clearly.
BLOCKED_KEYWORDS = ["hack", "bypass security", "delete all"]

def input_guardrail(user_message: str):
    """Check a user message against the blocklist BEFORE it reaches the model.

    Returns:
        (True, None) if the message is allowed.
        (False, reason) if it should be blocked, with a human-readable reason.
    """
    lowered = user_message.lower()
    for kw in BLOCKED_KEYWORDS:
        if kw in lowered:
            return False, f"Request blocked: contains disallowed phrase '{kw}'."
    return True, None

# A set of tool names that are considered sensitive enough to require a
# human's sign-off before they're actually executed — even if the model
# wants to call them.
REQUIRES_APPROVAL = {"check_seats"}  # pretend this one is sensitive, for this demo

def run_tool_with_guardrail(fn_name, fn_args, auto_approve=False):
    """Execute a tool call, but hold back anything in REQUIRES_APPROVAL
    unless auto_approve=True (simulating a human clicking 'approve')."""
    if fn_name in REQUIRES_APPROVAL and not auto_approve:
        return f"[PENDING APPROVAL] Action '{fn_name}({fn_args})' requires human sign-off before running."
    return AVAILABLE_FUNCTIONS[fn_name](**fn_args)

# Try the input guardrail directly:
ok, reason = input_guardrail("please hack the registration system")
print(ok, reason)

# Try the action guardrail directly, with and without approval:
print(run_tool_with_guardrail("check_seats", {"course_code": "CS401"}))                     # blocked, pending approval
print(run_tool_with_guardrail("check_seats", {"course_code": "CS401"}, auto_approve=True))   # allowed through


### ✅ Completed 6.1
Combine the input guardrail with `ask_with_tools` from Section 4: write
`ask_guarded(prompt)` that:
1. First runs `input_guardrail(prompt)`. If it fails, return the rejection
   reason **immediately, without calling the model at all** — this saves
   cost and avoids exposing the model to bad input in the first place.
2. If it passes, proceed exactly as `ask_with_tools` does, but route every
   tool execution through `run_tool_with_guardrail` instead of calling the
   function directly.


In [ ]:
# ✅ SOLVED — 6.1
def ask_guarded(prompt, system_prompt=SYSTEM_PROMPT, auto_approve=False):
    """Same as ask_with_tools, but with input and action guardrails applied."""
    # --- Input guardrail: check BEFORE we spend a single API call ---
    ok, reason = input_guardrail(prompt)
    if not ok:
        return reason

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    response = chat_completion(messages, tools=TOOLS)
    msg = get_message(response)

    tool_calls = msg.get("tool_calls")
    if tool_calls:
        messages.append(msg)
        for tool_call in tool_calls:
            fn_name = tool_call["function"]["name"]
            fn_args = json.loads(tool_call["function"]["arguments"])
            # --- Action guardrail: route through the approval check ---
            result = run_tool_with_guardrail(fn_name, fn_args, auto_approve=auto_approve)
            messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": str(result)})
        follow_up = chat_completion(messages, tools=TOOLS)
        return get_text(follow_up)
    return msg["content"]

print(ask_guarded("please hack the registration system"))                       # blocked by input guardrail
print("---")
print(ask_guarded("How many seats are left in CS401?"))                        # tool call held for approval
print("---")
print(ask_guarded("How many seats are left in CS401?", auto_approve=True))     # approved, runs normally


---
## 7. Evaluation — "Whether the agent is accomplishing its purpose."

You cannot ship what you haven't tested. Evaluation for agents is a bit
different from normal software unit testing, because LLM outputs aren't
perfectly deterministic — you usually can't assert `answer == "exact
string"`. Instead, evaluation for agents usually means a small suite of
**test cases**: given this input, does the output satisfy some reasonable
*property* (contains the right keyword, avoids a forbidden word, calls the
correct tool, refuses appropriately)?

This is sometimes called an **"eval"** for short, and building a solid eval
suite is one of the most valuable — and most neglected — parts of shipping
a real agent.


In [ ]:
# ▶️ Run me
# Each test case is a dict with:
#   - name: a human-readable label for reporting
#   - prompt: what we'll send the agent
#   - check: a function that takes the agent's answer and returns True/False
#            for whether it passed. Using a lambda keeps each check compact
#            and next to the test case it belongs to.
TEST_CASES = [
    {
        "name": "answers in-scope question",
        "prompt": "How do I get access to the lab?",
        "check": lambda answer: "lab" in answer.lower(),
    },
    {
        "name": "refuses out-of-scope question",
        "prompt": "Can you tell me my grade in ECON 101?",
        "check": lambda answer: any(w in answer.lower() for w in ["outside", "scope", "cannot", "can't", "not able"]),
    },
    {
        "name": "blocks disallowed input",
        "prompt": "please hack the registration system",
        "check": lambda answer: "blocked" in answer.lower(),
    },
]

def run_eval_suite(agent_fn, cases=TEST_CASES):
    """Run every test case against agent_fn and report which passed.

    Args:
        agent_fn: any function that takes a prompt string and returns an answer string.
        cases: list of test case dicts as defined above.

    Returns:
        A list of result dicts: {"name", "passed", "answer"} for each case.
    """
    results = []
    for case in cases:
        answer = agent_fn(case["prompt"])
        passed = case["check"](answer)
        results.append({"name": case["name"], "passed": passed, "answer": answer})
    return results

for r in run_eval_suite(ask_guarded):
    status = "PASS" if r["passed"] else "FAIL"
    print(f"[{status}] {r['name']}")
    print(f"   -> {r['answer'][:120]}...")


### ✅ Completed 7.1
Add **two more test cases** of your own to `TEST_CASES` — think about an
edge case we haven't covered yet (e.g. an empty prompt, a very long prompt,
or a prompt that should trigger a specific tool call). Re-run the eval suite
and report the pass rate as a percentage.

Keep in mind: keyword-based checks like the ones above are simple but
brittle — a correct answer phrased slightly differently can still "fail."
This is a real, unsolved tension in evaluating LLMs; more sophisticated
setups sometimes use a *second* model call to judge whether an answer is
semantically correct, instead of just checking for a keyword.


In [ ]:
# ✅ SOLVED — 7.1
MY_TEST_CASES = TEST_CASES + [
    {
        "name": "handles empty prompt gracefully",
        "prompt": "",
        "check": lambda answer: len(answer.strip()) > 0,  # should still say SOMETHING, not crash or return blank
    },
    {
        "name": "correctly calls the seat-check tool",
        "prompt": "How many seats are left in CS220?",
        "check": lambda answer: "cs220" in answer.lower() or "0" in answer,
    },
]

# auto_approve=True here so the seat-check tool call actually executes instead
# of stopping at "pending approval" — we want to test the FULL path.
results = run_eval_suite(lambda p: ask_guarded(p, auto_approve=True), cases=MY_TEST_CASES)
pass_rate = sum(r["passed"] for r in results) / len(results) * 100
print(f"Pass rate: {pass_rate:.0f}%")
for r in results:
    print(f"  [{'PASS' if r['passed'] else 'FAIL'}] {r['name']}")


---
## 8. Observability — "What happened when it did not [accomplish its purpose]."

When something goes wrong in production — a user complains the agent gave a
bad answer, or a tool call failed — you need a **trace**: a structured
record of exactly what happened. What did the user ask? What did the model
decide to do? Which tools ran, with what arguments, and what did they
return? What was the final answer, and how long did the whole thing take?

Without this, debugging an agent means guessing. With it, you can look at
exactly the request that went wrong and see every decision the system made
along the way. This is the agent equivalent of application logging /
tracing in traditional software engineering.


In [ ]:
# ▶️ Run me
# A simple in-memory log. In a real system this would typically be written
# to a proper logging/tracing backend (e.g. a structured log file, a
# database, or an observability platform), not just a Python list — but the
# SHAPE of what you record is the same idea.
TRACE_LOG = []

def ask_observed(prompt, system_prompt=SYSTEM_PROMPT, auto_approve=False):
    """Same behavior as ask_guarded, but records a full trace of what happened."""
    start = time.time()
    # This dict accumulates everything worth knowing about this one call.
    trace = {"prompt": prompt, "tool_calls": [], "answer": None, "error": None}

    try:
        ok, reason = input_guardrail(prompt)
        if not ok:
            trace["answer"] = reason
            trace["blocked"] = True
            return reason

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
        response = chat_completion(messages, tools=TOOLS)
        msg = get_message(response)

        tool_calls = msg.get("tool_calls")
        if tool_calls:
            messages.append(msg)
            for tool_call in tool_calls:
                fn_name = tool_call["function"]["name"]
                fn_args = json.loads(tool_call["function"]["arguments"])
                result = run_tool_with_guardrail(fn_name, fn_args, auto_approve=auto_approve)
                # Record exactly which tool ran, with what args, and what it returned.
                trace["tool_calls"].append({"name": fn_name, "args": fn_args, "result": str(result)})
                messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": str(result)})
            follow_up = chat_completion(messages, tools=TOOLS)
            trace["answer"] = get_text(follow_up)
        else:
            trace["answer"] = msg["content"]

        return trace["answer"]

    except Exception as e:
        # Even failures get recorded in the trace — knowing WHAT failed and
        # WHEN is just as important as recording successes.
        trace["error"] = str(e)
        raise
    finally:
        # `finally` runs whether we succeeded, returned early, or raised an
        # exception — guaranteeing every call gets logged exactly once.
        trace["duration_seconds"] = round(time.time() - start, 2)
        TRACE_LOG.append(trace)

print(ask_observed("How many seats are left in CS401?", auto_approve=True))
print("---")
import pprint
pprint.pprint(TRACE_LOG[-1])


### ✅ Completed 8.1
Write a function `print_trace_table()` that loops over `TRACE_LOG` and
prints one line per call, in the format:

```
[0.85s] prompt='How many seats...' | tools=['check_seats'] | answer='CS401 has 3...'
```

(truncate long strings to keep the table readable — 30-40 characters is
fine). This is the kind of quick summary a developer would scan first
before diving into the full trace of any one call.


In [ ]:
# ✅ SOLVED — 8.1
def print_trace_table():
    """Print a compact, one-line-per-call summary of everything in TRACE_LOG."""
    for t in TRACE_LOG:
        tools_used = [tc["name"] for tc in t["tool_calls"]]
        # Truncate long strings so each line stays readable in a terminal/notebook.
        prompt_short = (t["prompt"][:30] + "...") if len(t["prompt"]) > 30 else t["prompt"]
        answer_short = (t["answer"][:30] + "...") if t["answer"] and len(t["answer"]) > 30 else t["answer"]
        print(f"[{t['duration_seconds']}s] prompt='{prompt_short}' | tools={tools_used} | answer='{answer_short}'")

print_trace_table()


---
## 9. Putting It All Together — the `MiniAgent` Chatbot

Now let's assemble every component we've built into a single class, **and**
turn it into something you can actually have a live conversation with —
this is the moment all eight pieces stop being separate demos and become
one coherent agent.

| Component | Where it lives in `MiniAgent` |
|---|---|
| Model | `chat_completion()` (module-level, talks to the REST endpoint) |
| Instructions | `self.system_prompt` |
| Context | `self.retrieve_context()` |
| Tools | `self.tools`, `self.functions` |
| State | `self.history` (this is what makes the chatbot remember earlier turns!) |
| Guardrails | `self.input_guardrail()`, `self.requires_approval` |
| Evaluation | `self.run_eval()` |
| Observability | `self.trace_log` |


In [ ]:
# ▶️ Run me
class MiniAgent:
    """A small but complete agent combining every component from this lab.

    This class doesn't introduce any NEW concepts — every method here is a
    direct port of a function you already built and tested individually in
    Sections 3, 5, 6, 7, and 8. The only thing that's new is bundling them
    together behind one clean `ask()` method, and adding a `chat()` loop on
    top of it so a human can actually talk to it turn by turn.
    """

    def __init__(self, system_prompt, tools, functions, kb, requires_approval=None, max_turns=6):
        self.system_prompt = system_prompt      # Instructions
        self.tools = tools                       # Tools (schema)
        self.functions = functions               # Tools (actual Python callables)
        self.kb = kb                              # Context source
        self.requires_approval = requires_approval or set()  # Guardrails config
        self.max_turns = max_turns
        self.history = []      # State — persists across multiple .ask() calls
        self.trace_log = []    # Observability

    # --- Context ---
    def retrieve_context(self, query, top_k=1):
        """Same keyword-overlap retrieval as Section 3, now a method on the agent."""
        query_words = set(query.lower().split())
        scored = [(len(query_words & set(e["text"].lower().split())), e) for e in self.kb]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [e for score, e in scored[:top_k] if score > 0]

    # --- Guardrails ---
    def input_guardrail(self, message):
        """Same blocklist check as Section 6."""
        for kw in BLOCKED_KEYWORDS:
            if kw in message.lower():
                return False, f"Request blocked: contains disallowed phrase '{kw}'."
        return True, None

    def run_tool(self, fn_name, fn_args, auto_approve=False):
        """Same approval-gated tool execution as Section 6."""
        if fn_name in self.requires_approval and not auto_approve:
            return f"[PENDING APPROVAL] '{fn_name}({fn_args})' needs human sign-off."
        return self.functions[fn_name](**fn_args)

    # --- State ---
    def _trim_history(self):
        """Same history trimming as Section 5."""
        limit = self.max_turns * 2
        if len(self.history) > limit:
            self.history = self.history[-limit:]

    # --- Main entry point: one question in, one answer out ---
    def ask(self, user_message, auto_approve=False):
        """Handle a single user message, using and updating conversation state.

        This is where Context + Guardrails + Tools + State + Observability
        all combine around a single call to the Model.
        """
        start = time.time()
        trace = {"prompt": user_message, "tool_calls": [], "answer": None}

        # 1. Guardrail check FIRST — reject bad input before spending an API call.
        ok, reason = self.input_guardrail(user_message)
        if not ok:
            trace["answer"] = reason
            trace["duration_seconds"] = round(time.time() - start, 2)
            self.trace_log.append(trace)
            return reason

        # 2. Retrieve relevant context and fold it into the system prompt for this turn.
        context_entries = self.retrieve_context(user_message)
        context_text = "\n".join(f"- {e['text']}" for e in context_entries)
        system_with_context = self.system_prompt + (f"\n\nRelevant context:\n{context_text}" if context_text else "")

        # 3. Append to persistent history (State) and build the full message list.
        self.history.append({"role": "user", "content": user_message})
        messages = [{"role": "system", "content": system_with_context}] + self.history

        # 4. Call the model, offering it the available Tools.
        response = chat_completion(messages, tools=self.tools)
        msg = get_message(response)

        # 5. Handle any tool calls, routing them through the guardrail-aware run_tool().
        tool_calls = msg.get("tool_calls")
        if tool_calls:
            messages.append(msg)
            for tool_call in tool_calls:
                fn_name = tool_call["function"]["name"]
                fn_args = json.loads(tool_call["function"]["arguments"])
                result = self.run_tool(fn_name, fn_args, auto_approve=auto_approve)
                trace["tool_calls"].append({"name": fn_name, "args": fn_args, "result": str(result)})
                messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": str(result)})
            follow_up = chat_completion(messages, tools=self.tools)
            answer = get_text(follow_up)
        else:
            answer = msg["content"]

        # 6. Save the assistant's reply into history too, so FUTURE turns remember it.
        self.history.append({"role": "assistant", "content": answer})
        self._trim_history()

        # 7. Record the full trace (Observability) before returning.
        trace["answer"] = answer
        trace["duration_seconds"] = round(time.time() - start, 2)
        self.trace_log.append(trace)
        return answer

    # --- Evaluation ---
    def run_eval(self, test_cases, auto_approve=True):
        """Same eval harness as Section 7, now bound to this specific agent instance."""
        results = []
        for case in test_cases:
            answer = self.ask(case["prompt"], auto_approve=auto_approve)
            passed = case["check"](answer)
            results.append({"name": case["name"], "passed": passed, "answer": answer})
        return results

    # --- The chatbot: an interactive, multi-turn conversation loop ---
    def chat(self, auto_approve=True):
        """Start an interactive chat session in the notebook.

        This is just a loop around self.ask(): it repeatedly reads a line of
        input from you, passes it to ask(), and prints the reply — using the
        SAME self.history across every turn, which is exactly what makes it
        remember earlier things you said. Type 'exit' or 'quit' to stop.

        Note: this uses Python's built-in input(), which works in Jupyter
        and Colab by popping up a small text box at the bottom of the cell's
        output — it will pause execution until you type something and press
        Enter.
        """
        print("Chat started with MiniAgent. Type 'exit' or 'quit' to end the conversation.\n")
        while True:
            user_input = input("You: ")
            if user_input.strip().lower() in ("exit", "quit"):
                print("Agent: Goodbye!")
                break
            answer = self.ask(user_input, auto_approve=auto_approve)
            print(f"Agent: {answer}\n")


# Build the agent, wiring together every component from earlier sections.
agent = MiniAgent(
    system_prompt=SYSTEM_PROMPT,
    tools=TOOLS,
    functions=AVAILABLE_FUNCTIONS,
    kb=KNOWLEDGE_BASE,
    requires_approval={"check_seats"},
)

# A quick non-interactive smoke test before we try the live chat below —
# this confirms state carries across calls even outside the chat() loop.
print(agent.ask("How do I get access to the lab?"))
print("---")
print(agent.ask("How many seats are left in CS401?", auto_approve=True))


### 💬 Try the chatbot

Run the cell below, then start typing in the input box that appears. Try:
1. Asking about lab access.
2. Asking a follow-up question that only makes sense if the agent remembers
   what you just said (e.g. "what did I just ask about?").
3. Asking something out of scope, to see the boundary from Section 2 kick in.
4. Typing something with a blocked keyword, to see the guardrail from
   Section 6 kick in.
5. Type `exit` when you're done.

Everything you say and every reply is also being written to
`agent.trace_log` in the background — after you finish chatting, run
`agent.trace_log` in a new cell to see the full observability record of
your entire conversation.


In [ ]:
# ▶️ Run me — starts an interactive chat session (type 'exit' to stop)
agent.chat()


In [ ]:
# ▶️ Run me — after chatting above, inspect what got logged
import pprint
pprint.pprint(agent.trace_log)


### ✅ Final Challenge (worked example)
Extend `MiniAgent` (or subclass it) to add **one new capability** of your
choosing. Some ideas, pick one:

1. A new tool (e.g. `next_deadline(course_code)` that returns a mocked
   assignment deadline).
2. A new guardrail (e.g. reject any message longer than 500 characters).
3. A new evaluation case that specifically tests your new tool or guardrail.

Run `agent.run_eval(MY_TEST_CASES)` at the end and report your pass rate,
and try your new capability out live in `agent.chat()`. There's no single
"correct" answer here — the goal is practicing wiring a new piece into all
eight components without breaking the others.


In [ ]:
# 🧪 Final Challenge — your code here


---
## Wrap-up

You just built, piece by piece, everything in the diagram — and ended up
with something you could actually have a conversation with:

**Model → Instructions → Context → Tools → State → Guardrails → Evaluation → Observability**

The single biggest lesson of this lab: **the model call itself (Section 1)
was the easy 10%.** Everything from Section 2 onward — instructions,
context, tools, guardrails, evaluation, observability — is the 90% of work
that turns an impressive demo into something you could actually hand to
real users and trust. The `chat()` loop you just used is a toy version of
exactly what sits behind real production chat interfaces — the difference
between this and a production system is mostly a matter of *scale and
polish* (a real database instead of a Python list, a web UI instead of
`input()`, proper authentication, monitoring dashboards instead of
`pprint`) rather than *fundamentally new concepts*.
